# Demo: dạy robot KETI mở ngăn kéo bằng VLA — từ mô phỏng tới model

**Mục tiêu dự án.** Robot (xe + cột nâng + tay xArm7 + gripper) nghe một câu lệnh như *"open the top drawer"*, nhìn
qua 2 camera, rồi tự điều khiển cánh tay mở đúng ngăn kéo. Bộ não là một **VLA** (Vision–Language–Action model):
đầu vào là **ảnh + câu lệnh**, đầu ra là **góc khớp cánh tay**, 30 lần mỗi giây.

**Vì sao cần mô phỏng?** Model cần hàng trăm lần làm mẫu. Làm trên robot thật thì chậm và tốn công;
trong mô phỏng (MuJoCo) máy tính tự làm mẫu, tự đổi cảnh, và 500 lần làm mẫu chỉ mất khoảng 1 giờ.

| Bước | Nội dung | Thời gian |
|---|---|---|
| 0 | Kiểm tra môi trường | 10 s |
| 1 | Robot trong mô phỏng | 1 phút |
| 2 | Làm cảnh sim giống cảnh thật | 1 phút |
| 3 | "Người thầy" (expert) mở ngăn kéo | 1 phút |
| 4 | Sinh dữ liệu huấn luyện (bản mini) | 3–4 phút |
| 5 | Huấn luyện model (bản mini) | 3–5 phút |
| 6 | **Chạy model đã huấn luyện với câu lệnh tuỳ chọn** | 1–2 phút |
| 7 | Kết quả và bước tiếp theo | 10 s |

**Cách chạy:** chọn kernel **"kaair demo (host)"** (góc trên bên phải), rồi chạy lần lượt từng ô (`Shift+Enter`).
Mọi lệnh mô phỏng chạy trong container `kaair_sim_humble`; **không có lệnh nào gửi tới robot thật**.

In [2]:
import sys, os
sys.path.insert(0, os.path.expanduser('~/trung_workdir/kaair_sim/docs/demo'))
from demo_lib import *

## Bước 0 — Kiểm tra môi trường
Mọi thứ chạy trên server A6000: một **container Docker** (ROS 2 + MuJoCo + PyTorch/LeRobot), dữ liệu và model
lưu trên **NAS** để không đầy ổ cứng máy. GPU 1, 2 thường có việc của người khác nên demo chỉ dùng GPU 0 và 3.

In [3]:
status()

| | |
|---|---|
| container `kaair_sim_humble` | running |
| GPU 0 (ours) | 5.2 / 48 GB, 0% busy |
| GPU 1 (other user) | 0.6 / 48 GB, 0% busy |
| GPU 2 (other user) | 3.0 / 48 GB, 0% busy |
| GPU 3 (ours) | 20.7 / 48 GB, 78% busy |
| NAS datasets | ok (`/mnt/nas/lerobot_shared/kaair_sim/datasets`) |
| NAS checkpoints | ok (`/mnt/nas/lerobot_shared/kaair_sim/checkpoints`) |
| trained model | 005000 |

## Bước 1 — Robot trong mô phỏng
Repo ROS 2 của robot có sẵn file **URDF**: bản mô tả hình học (các khâu, các khớp). Ta chuyển nó sang
**MuJoCo** và bổ sung phần vật lý: khối lượng, va chạm, "động cơ" ở mỗi khớp. Kết quả: robot trong máy tính
chuyển động và cầm nắm được như robot thật.

Ô dưới đặt robot ở **tư thế chuẩn bị mở tủ** — y như robot thật dùng (`moveh::left`, `lift::0.5`,
`movej::approach_drawer` trong phần mềm kcare) — rồi chụp 3 ảnh: toàn cảnh, camera đầu, camera tay.

In [ ]:
sh('python3 scripts/drawer_ready.py', keep=r'mode=|TCP|handle')
show_img('/sim/out/open_drawer/ready_left_overview.png', 800)
show_img('/sim/out/open_drawer/ready_left_head.png', 560); show_img('/sim/out/open_drawer/ready_left_wrist.png', 560)

## Bước 2 — Làm cảnh sim giống cảnh thật
Model học **từ ảnh**. Nếu ảnh sim khác xa ảnh thật, model học trong sim sẽ "bỡ ngỡ" trên robot. Ta chụp một cảnh
từ robot thật (chỉ đọc camera, không cử động robot) rồi chỉnh sim theo 3 lớp:
- **hình học**: tủ đặt ở đâu, đầu robot cúi bao nhiêu độ (đo từ ảnh thật);
- **bề mặt**: vân gỗ, sàn, phòng làm việc — cắt từ chính ảnh robot chụp;
- **camera**: mô phỏng camera Femto (đầu) và RealSense D405 (tay): méo ống kính, tự cân sáng, màu, nhiễu, nén JPEG.

Trái: ảnh **thật**. Phải: ảnh **sim** ở đúng tư thế robot lúc chụp. (Chạy lại mất ~1 phút; bỏ comment dòng đầu.)

In [ ]:
# sh('python3 scripts/render_real_compare.py /nas/real/20260928_scene1')
show_img('/sim/out/real_compare/compare.png', 1000)

## Bước 3 — "Người thầy" (expert) mở ngăn kéo
Để có dữ liệu mẫu, ta viết một chương trình **biết trước đáp án**: nó biết chính xác tay nắm ở đâu (thông tin
"gian lận" mà robot thật không có) và tính đường đi cho cánh tay: tiến tới → kẹp tay nắm → kéo 25 cm → nhả → lùi.

Model sau này **không** được biết toạ độ đó — nó phải tự suy ra từ ảnh.

👉 **Thử:** đổi `DRAWER` thành `'top'`, `'middle'` hoặc `'bottom'`; đổi `SEED` để có cảnh khác (vị trí tủ, ánh sáng,
độ cao lift, góc đầu).

In [ ]:
DRAWER, SEED = 'top', 7
sh(f'python3 scripts/demo.py expert --drawer {DRAWER} --seed {SEED}', keep=r'expert|->')
show_video(f'/sim/out/demo/expert_{DRAWER}_seed{SEED}.mp4')

## Bước 4 — Sinh dữ liệu huấn luyện (bản mini: 6 lần làm mẫu)
Chạy expert nhiều lần, **mỗi lần một cảnh ngẫu nhiên** và **một câu lệnh ngẫu nhiên** (ngăn trên / giữa / dưới,
nhiều cách nói: *"open the 1st drawer"*, *"open the lowest drawer"*…). Mỗi 1/30 giây ghi lại:
- robot **thấy** gì: 2 ảnh + 8 số trạng thái (7 góc khớp tay + độ mở gripper);
- robot **làm** gì: 8 số lệnh cho bước kế tiếp;
- **câu lệnh** của lần đó.

Định dạng **LeRobot**, giống hệt định dạng dữ liệu thu trên robot thật, nên sau này trộn chung được.
Bản thật có 500 lần làm mẫu (168 000 khung hình); ở đây chỉ 6.

In [ ]:
GPU = pick_gpu()
sh('rm -rf /nas/datasets/_demo_mini /nas/datasets/_demo_mini_episodes.jsonl /nas/datasets/_shards/_demo_mini', quiet=True)
sh(f'.venv/bin/python -W ignore scripts/gen_dataset.py --n 6 --workers 3 --gpu-ids {GPU} --seed 42 '
   '--repo-id _demo_mini --task /sim/config/task_open_drawer.yaml '
   '--rand-config /sim/config/randomization_real.yaml --camera-model', keep=r'\] w\d|done in')

Xem bên trong dataset: một khung hình gồm những gì, và các câu lệnh đã sinh.

In [ ]:
sh('''.venv/bin/python -W ignore - <<'EOF'
from lerobot.datasets.lerobot_dataset import LeRobotDataset
ds = LeRobotDataset('_demo_mini', root='/nas/datasets/_demo_mini', video_backend='pyav')
print(f'{ds.meta.total_episodes} episodes, {ds.meta.total_frames} frames, {ds.meta.fps} frames/s')
f = ds[0]
for k, v in f.items():
    if hasattr(v, 'shape'): print(f'  {k:28s} shape {tuple(v.shape)}')
print('  state  =', [round(x, 3) for x in f['observation.state'].tolist()])
print('  action =', [round(x, 3) for x in f['action'].tolist()])
print('prompts:', sorted({ds[int(ds.meta.episodes[e]["dataset_from_index"])]['task'] for e in range(ds.meta.total_episodes)}))
EOF''', keep=r'^(?!.*(Warning|warn|it/s)).*$')

## Bước 5 — Huấn luyện model (bản mini: 200 bước)
Model là **SmolVLA** (450 triệu tham số) của Hugging Face, đã được huấn luyện trước trên nhiều robot khác. Ta
**tinh chỉnh** (fine-tune) nó trên dữ liệu của mình: cho xem (ảnh, câu lệnh, trạng thái) và bắt đoán hành động
của expert; **loss** = mức sai lệch giữa hành động model đoán và hành động expert. Loss giảm = model đang học.

Bản mini dưới đây chỉ để thấy quá trình chạy và loss giảm — **200 bước trên 6 episode chưa đủ để mở được tủ**.
Model thật: 20 000 bước trên 500 episode, ~8 giờ trên 1 GPU.

In [ ]:
GPU = pick_gpu()
OUT = '/nas/checkpoints/_demo_mini_train'
sh(f'rm -rf {OUT}', quiet=True)
log = sh(f'''CUDA_VISIBLE_DEVICES={GPU} .venv/bin/lerobot-train \
  --policy.path=/nas/checkpoints/_bases/smolvla_base_kaair --policy.push_to_hub=false \
  --rename_map='{{"observation.images.head":"observation.images.camera1","observation.images.wrist":"observation.images.camera2"}}' \
  --dataset.repo_id=_demo_mini --dataset.root=/nas/datasets/_demo_mini --dataset.video_backend=pyav \
  --output_dir={OUT} --job_name=demo_mini --batch_size=8 --steps=200 --save_freq=200 --log_freq=10 \
  --num_workers=4 --wandb.enable=false''', keep=r'step:|num_learnable|Error', ret=True)
plot_loss(log, 'mini training (200 steps)')

So sánh: đường loss của **lượt huấn luyện thật** (500 episode, đang chạy / đã xong).

In [ ]:
plot_loss(open(host('/sim/out/train_real500.log'), errors='ignore').read(), 'real training run (500 episodes)')

## Bước 6 — Chạy model đã huấn luyện
Giờ model **chỉ** nhận ảnh từ 2 camera + câu lệnh — không biết tay nắm ở đâu, không có expert. Nó tự quyết định
cánh tay đi thế nào, 30 lần mỗi giây. Video: trên là camera toàn cảnh (chỉ để người xem, model không thấy),
dưới là 2 ảnh model thực sự nhận được.

👉 **Thử:** đổi `PROMPT` — ví dụ *"open the 2nd drawer"*, *"open the lowest drawer"*, hoặc một câu model chưa từng
thấy như *"open the drawer at the bottom"*. Đổi `SEED` để có cảnh khác.

In [ ]:
PROMPT, SEED = 'open the top drawer', 11
CKPT = latest_ckpt()
print('model:', os.path.realpath(host(CKPT)))
sh(f'CUDA_VISIBLE_DEVICES={pick_gpu()} .venv/bin/python -W ignore scripts/demo.py run --ckpt {CKPT} '
   f'--prompt "{PROMPT}" --seed {SEED}', keep=r'SUCCESS|FAIL|WRONG|opened|->')
slug = re.sub(r'[^a-z0-9]+', '_', PROMPT.lower()).strip('_')[:40]
show_video(f'/sim/out/demo/run_{slug}_seed{SEED}.mp4')

## Bước 7 — Kết quả và bước tiếp theo
Mỗi checkpoint (5k, 10k, 15k, 20k bước) được tự động thử trên **50 cảnh mới** (chưa từng thấy khi huấn luyện).
Thành công = mở **đúng** ngăn được yêu cầu ≥ 20 cm; mở nhầm ngăn khác = thất bại.

| Mốc trước đó | Kết quả |
|---|---|
| Model 1 ngăn, 1 câu lệnh (cảnh cũ) | **98%** thành công |
| Model nhiều ngăn, cảnh giống thật | bảng dưới (cập nhật khi huấn luyện chạy) |

In [ ]:
display(eval_table())

Vài clip thành công / thất bại (checkpoint 5 000 bước — thất bại điển hình: tay tới trước tủ nhưng kẹp hụt tay
nắm, rồi đẩy ngăn vào thay vì kéo ra).

In [ ]:
import glob
for f in sorted(glob.glob(host('/sim/out/eval_clips/*_ok_*.mp4')))[:1] + sorted(glob.glob(host('/sim/out/eval_clips/*_fail_*.mp4')))[:1]:
    print(os.path.basename(f)); show_video(f.replace(host('/sim'), '/sim'), 700)

**Bước tiếp theo**
1. Hoàn tất huấn luyện, chọn checkpoint tốt nhất theo bảng trên.
2. Chạy thử trên robot thật (có người giám sát, nút Stop luôn sẵn sàng).
3. Thu thêm một ít dữ liệu thật bằng skill `open_drawer` của robot, huấn luyện chung với dữ liệu sim (*co-training*).

Tài liệu chi tiết: `docs/kaair_sim_guide.pdf`.

### Dọn dẹp sau demo (xoá dữ liệu và model mini trên NAS)

In [ ]:
sh('rm -rf /nas/datasets/_demo_mini /nas/datasets/_demo_mini_episodes.jsonl /nas/datasets/_shards/_demo_mini '
   '/nas/checkpoints/_demo_mini_train && echo cleaned')